# Hot Cities, Blind Sensors — Heat Where the Map Is Empty

**Standalone.** Synthetic city LST + sparse stations; metrics for heat ∩ missingness.

VI: Demo synthetic; REAL_DATA = OpenAQ + MODIS/ERA5 (xem README).

In [ ]:
%pip install -q numpy pandas matplotlib scipy

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
RNG = np.random.default_rng(42)
GRID, N_STATIONS, RADIUS, HEAT_PCT = 64, 12, 6.0, 90

def make_city_lst(n=GRID):
    yy, xx = np.mgrid[0:n, 0:n]
    core = 32 + 8 * np.exp(-((xx - n*0.45)**2 + (yy - n*0.5)**2) / (2*12**2))
    industrial = 5 * np.exp(-((xx - n*0.75)**2 + (yy - n*0.25)**2) / (2*6**2))
    park = -4 * np.exp(-((xx - n*0.3)**2 + (yy - n*0.7)**2) / (2*8**2))
    return gaussian_filter(core + industrial + park + RNG.normal(0, 0.4, (n, n)), 1.0)

def place_stations(n, k):
    coords = []
    for _ in range(k):
        x = int(np.clip(RNG.normal(n*0.4, n*0.12), 0, n-1))
        y = int(np.clip(RNG.normal(n*0.55, n*0.12), 0, n-1))
        coords.append((y, x))
    return np.array(coords)

def coverage_mask(shape, stations, radius):
    h, w = shape; yy, xx = np.mgrid[0:h, 0:w]
    covered = np.zeros(shape, dtype=bool)
    for y, x in stations:
        covered |= (yy - y)**2 + (xx - x)**2 <= radius**2
    return covered


In [ ]:
lst = make_city_lst()
stations = place_stations(GRID, N_STATIONS)
covered = coverage_mask(lst.shape, stations, RADIUS)
thr = np.percentile(lst, HEAT_PCT)
heat, blind = lst >= thr, ~covered
blind_heat = heat & blind
frac = float(blind_heat.sum() / max(heat.sum(), 1))

# null
fracs = []
for _ in range(200):
    ys, xs = RNG.integers(0, GRID, N_STATIONS), RNG.integers(0, GRID, N_STATIONS)
    cov = coverage_mask(lst.shape, np.column_stack([ys, xs]), RADIUS)
    fracs.append(((heat & ~cov).sum()) / max(heat.sum(), 1))
null_mean = float(np.mean(fracs))

wards = np.zeros((GRID, GRID), dtype=int)
bs = GRID // 4; wid = 0
for i in range(4):
    for j in range(4):
        wards[i*bs:(i+1)*bs, j*bs:(j+1)*bs] = wid; wid += 1
rows = []
for w in range(16):
    m = wards == w
    rows.append(dict(ward=w, mean_lst=float(lst[m].mean()), frac_blind=float(blind[m].mean()),
                     frac_blind_and_heat=float((blind_heat & m).sum()/m.sum()),
                     hot_blind_score=float(lst[m].mean()*blind[m].mean())))
df = pd.DataFrame(rows).sort_values('hot_blind_score', ascending=False)
df.to_csv(RESULTS/'heat_missingness.csv', index=False)
summary = dict(mode='SYNTHETIC', frac_heat_cells_uncovered=frac, null_mean=null_mean,
               excess_vs_null=frac-null_mean)
(RESULTS/'summary.json').write_text(json.dumps(summary, indent=2))
print(summary); display(df.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
im0 = axes[0].imshow(lst, cmap='inferno')
axes[0].scatter(stations[:,1], stations[:,0], c='cyan', s=40, marker='^')
axes[0].set_title('LST + stations'); plt.colorbar(im0, ax=axes[0], fraction=0.046)
axes[1].imshow(heat, cmap='Reds'); axes[1].set_title('Heat extreme')
axes[2].imshow(blind_heat, cmap='Purples'); axes[2].set_title('Blind ∩ heat')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); fig.savefig(RESULTS/'city_map.png', dpi=120); plt.show()


## Multi-seed audit (seeds 42–51)
Same logic as `src/run_experiment.py`: each seed uses its own `default_rng(seed)`; writes `results/multi_seed.csv` and `results/summary_multi.json` (Tables 1 & 3 of the paper).

In [ ]:
SEEDS = list(range(42, 52)); N_PERM = 200

def run_one(seed):
    rng = np.random.default_rng(seed)
    n = GRID
    yy, xx = np.mgrid[0:n, 0:n]
    core = 32 + 8 * np.exp(-((xx - n*0.45)**2 + (yy - n*0.5)**2) / (2*12**2))
    industrial = 5 * np.exp(-((xx - n*0.75)**2 + (yy - n*0.25)**2) / (2*6**2))
    park = -4 * np.exp(-((xx - n*0.3)**2 + (yy - n*0.7)**2) / (2*8**2))
    lst = gaussian_filter(core + industrial + park + rng.normal(0, 0.4, (n, n)), 1.0)
    st = []
    for _ in range(N_STATIONS):
        x = int(np.clip(int(rng.normal(n*0.4, n*0.12)), 0, n-1))
        y = int(np.clip(int(rng.normal(n*0.55, n*0.12)), 0, n-1))
        st.append((y, x))
    blind = ~coverage_mask(lst.shape, np.array(st), RADIUS)
    heat = lst >= np.percentile(lst, HEAT_PCT)
    frac = float((heat & blind).sum() / max(heat.sum(), 1))
    fr = []
    for _ in range(N_PERM):
        ys, xs = rng.integers(0, n, size=N_STATIONS), rng.integers(0, n, size=N_STATIONS)
        cov = coverage_mask(heat.shape, np.column_stack([ys, xs]), RADIUS)
        fr.append((heat & ~cov).sum() / max(heat.sum(), 1))
    null = float(np.mean(fr))
    scores = [float(lst[wards == w].mean() * blind[wards == w].mean()) for w in range(16)]
    tw = int(np.argmax(scores))
    return dict(seed=seed, frac_heat_cells_uncovered=frac, null_mean_frac_heat_uncovered=null,
                excess_vs_null=frac - null, top_ward=tw, top_ward_score=scores[tw])

sdf = pd.DataFrame([run_one(s) for s in SEEDS])
sdf.to_csv(RESULTS/'multi_seed.csv', index=False)
multi = dict(excess_mean_10seeds=float(sdf.excess_vs_null.mean()),
             excess_std_10seeds=float(sdf.excess_vs_null.std(ddof=1)),
             frac_heat_uncovered_mean_10seeds=float(sdf.frac_heat_cells_uncovered.mean()),
             frac_heat_uncovered_std_10seeds=float(sdf.frac_heat_cells_uncovered.std(ddof=1)),
             frac_seeds_excess_negative=float((sdf.excess_vs_null < 0).mean()),
             top_ward_counts=sdf.top_ward.value_counts().to_dict())
(RESULTS/'summary_multi.json').write_text(json.dumps(multi, indent=2, default=int))
print(sdf.round(4).to_string(index=False)); print(json.dumps(multi, indent=2, default=int))

## v4 extensions (= `src/run_extended.py`, pasted verbatim)
Placement scenarios (core-biased / uniform / periphery-biased) as a positive control, Monte Carlo p-values, 100-seed calibration, one-at-a-time sensitivity (k, r, percentile), and a min-max normalised ward score.

In [ ]:
#!/usr/bin/env python3
"""Hot Cities v4 extensions: placement scenarios (positive control), Monte Carlo
p-values, null calibration, one-at-a-time sensitivity, normalised ward score.

Self-contained (also pasted verbatim into the Kaggle notebook). The "core"
scenario at the baseline config reproduces results/multi_seed.csv exactly:
same RNG call order as src/run_experiment.py.
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:  # running inside a notebook
    ROOT = Path(".")
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

GRID = 64
BASE = dict(k=12, r=6.0, pct=90)
N_PERM = 200
SEEDS = list(range(42, 52))          # narrative seeds, as in the paper
CAL_SEEDS = list(range(42, 142))     # 100 seeds for calibration / power
SCENARIOS = ["core", "uniform", "periphery"]
CORE_C = (0.45, 0.50)                # urban-core centre (x, y) as fraction of n
PERIPH_MIN_DIST = 0.30               # periphery stations: > 0.30n from core centre

_n = GRID
_YY, _XX = np.mgrid[0:_n, 0:_n]


def make_city_lst(rng, n=GRID):
    yy, xx = np.mgrid[0:n, 0:n]
    core = 32 + 8 * np.exp(-((xx - n * 0.45) ** 2 + (yy - n * 0.5) ** 2) / (2 * 12**2))
    industrial = 5 * np.exp(-((xx - n * 0.75) ** 2 + (yy - n * 0.25) ** 2) / (2 * 6**2))
    park_cool = -4 * np.exp(-((xx - n * 0.3) ** 2 + (yy - n * 0.7) ** 2) / (2 * 8**2))
    noise = rng.normal(0, 0.4, size=(n, n))
    return gaussian_filter(core + industrial + park_cool + noise, sigma=1.0)


def place_core(rng, n, k):
    coords = []
    for _ in range(k):
        x = int(rng.normal(n * 0.4, n * 0.12))
        y = int(rng.normal(n * 0.55, n * 0.12))
        coords.append((int(np.clip(y, 0, n - 1)), int(np.clip(x, 0, n - 1))))
    return np.array(coords, dtype=int)


def place_uniform(rng, n, k):
    return np.column_stack([rng.integers(0, n, size=k), rng.integers(0, n, size=k)])


def place_periphery(rng, n, k):
    coords = []
    while len(coords) < k:
        y, x = int(rng.integers(0, n)), int(rng.integers(0, n))
        if np.hypot(x - n * CORE_C[0], y - n * CORE_C[1]) > PERIPH_MIN_DIST * n:
            coords.append((y, x))
    return np.array(coords, dtype=int)


PLACERS = dict(core=place_core, uniform=place_uniform, periphery=place_periphery)


def coverage_mask(stations, r):
    s = np.asarray(stations)
    d2 = (_YY[None] - s[:, 0, None, None]) ** 2 + (_XX[None] - s[:, 1, None, None]) ** 2
    return (d2 <= r**2).any(axis=0)


WARDS = np.zeros((_n, _n), dtype=int)
_bs = _n // 4
for _i in range(4):
    for _j in range(4):
        WARDS[_i * _bs:(_i + 1) * _bs, _j * _bs:(_j + 1) * _bs] = _i * 4 + _j


def audit(seed, scenario="core", k=12, r=6.0, pct=90, n_perm=N_PERM):
    rng = np.random.default_rng(seed)
    lst = make_city_lst(rng)
    stations = PLACERS[scenario](rng, GRID, k)
    blind = ~coverage_mask(stations, r)
    heat = lst >= np.percentile(lst, pct)
    nh = max(heat.sum(), 1)
    b_h = float((heat & blind).sum() / nh)
    null = np.empty(n_perm)
    for i in range(n_perm):
        ys = rng.integers(0, GRID, size=k)
        xs = rng.integers(0, GRID, size=k)
        null[i] = (heat & ~coverage_mask(np.column_stack([ys, xs]), r)).sum() / nh
    # wards: raw score (paper) and min-max normalised LST score
    lw = np.array([lst[WARDS == w].mean() for w in range(16)])
    fb = np.array([blind[WARDS == w].mean() for w in range(16)])
    raw = lw * fb
    norm = (lw - lw.min()) / (lw.max() - lw.min()) * fb
    return dict(
        seed=seed, scenario=scenario, k=k, r=r, pct=pct,
        B_h=b_h, null_mean=float(null.mean()),
        null_q025=float(np.quantile(null, 0.025)), null_q975=float(np.quantile(null, 0.975)),
        E=b_h - float(null.mean()),
        p_upper=float((1 + (null >= b_h).sum()) / (n_perm + 1)),  # heat MORE blind than chance
        p_lower=float((1 + (null <= b_h).sum()) / (n_perm + 1)),  # heat LESS blind than chance
        top_ward_raw=int(raw.argmax()), top_ward_norm=int(norm.argmax()),
        _ward=dict(lw=lw, fb=fb, raw=raw, norm=norm),
    )


def strip(d):
    return {k: v for k, v in d.items() if not k.startswith("_")}


def main():
    # 1) scenarios x narrative seeds
    sc = [audit(s, sn) for sn in SCENARIOS for s in SEEDS]
    sc_df = pd.DataFrame([strip(d) for d in sc])
    sc_df.to_csv(RESULTS / "ext_scenarios.csv", index=False)

    # 2) calibration / power over 100 seeds
    cal = pd.DataFrame([strip(audit(s, sn)) for sn in SCENARIOS for s in CAL_SEEDS])
    cal.to_csv(RESULTS / "ext_calibration_raw.csv", index=False)
    cal_sum = cal.groupby("scenario").agg(
        E_mean=("E", "mean"), E_sd=("E", "std"),
        frac_E_pos=("E", lambda e: float((e > 0).mean())),
        rate_p_upper_05=("p_upper", lambda p: float((p < 0.05).mean())),
        rate_p_lower_05=("p_lower", lambda p: float((p < 0.05).mean())),
    ).reindex(SCENARIOS)
    cal_sum.to_csv(RESULTS / "ext_calibration.csv")

    # 3) one-at-a-time sensitivity (core + periphery, narrative seeds)
    grid = [("baseline", None)] + [("k", v) for v in (6, 24)] + [("r", v) for v in (3.0, 9.0)] \
        + [("pct", v) for v in (80, 95)]
    rows = []
    for factor, val in grid:
        cfg = dict(BASE) if val is None else {**BASE, factor: val}
        for sn in ("core", "periphery"):
            es = np.array([audit(s, sn, **cfg)["E"] for s in SEEDS])
            rows.append(dict(factor=factor, value=cfg[factor] if val is not None else "-",
                             scenario=sn, E_mean=es.mean(), E_sd=es.std(ddof=1),
                             frac_E_neg=float((es < 0).mean()), frac_E_pos=float((es > 0).mean())))
    sens = pd.DataFrame(rows)
    sens.to_csv(RESULTS / "ext_sensitivity.csv", index=False)

    # 4) normalised ward score, seed 42 core
    w42 = next(d for d in sc if d["seed"] == 42 and d["scenario"] == "core")["_ward"]
    wdf = pd.DataFrame(dict(ward=range(16), mean_lst=w42["lw"], frac_blind=w42["fb"],
                            S_raw=w42["raw"], S_norm=w42["norm"])).sort_values("S_norm", ascending=False)
    wdf.to_csv(RESULTS / "ext_wards_seed42.csv", index=False)

    core = sc_df[sc_df.scenario == "core"]
    summary = dict(
        mode="SYNTHETIC_city_grid_v4_extensions",
        n_perm=N_PERM, seeds=SEEDS, calibration_seeds=f"{CAL_SEEDS[0]}-{CAL_SEEDS[-1]}",
        periphery_min_dist_frac=PERIPH_MIN_DIST,
        scenarios_10seeds={sn: dict(
            E_mean=float(g.E.mean()), E_sd=float(g.E.std(ddof=1)),
            B_h_mean=float(g.B_h.mean()), frac_E_pos=float((g.E > 0).mean()),
            n_p_upper_05=int((g.p_upper < 0.05).sum()), n_p_lower_05=int((g.p_lower < 0.05).sum()))
            for sn, g in sc_df.groupby("scenario")},
        calibration_100seeds=cal_sum.reset_index().to_dict(orient="records"),
        core_top_ward_raw_counts={int(k): int(v) for k, v in core.top_ward_raw.value_counts().items()},
        core_top_ward_norm_counts={int(k): int(v) for k, v in core.top_ward_norm.value_counts().items()},
    )
    (RESULTS / "ext_summary.json").write_text(json.dumps(summary, indent=2))

    # figure: E by scenario over 100 seeds
    fig, ax = plt.subplots(figsize=(5.5, 3.0))
    rng = np.random.default_rng(0)
    for i, sn in enumerate(SCENARIOS):
        e = cal.loc[cal.scenario == sn, "E"].to_numpy()
        ax.scatter(e, i + rng.uniform(-0.15, 0.15, e.size), s=10, alpha=0.6)
        ax.plot([e.mean()] * 2, [i - 0.3, i + 0.3], color="black", lw=1.5)
    ax.axvline(0, color="gray", ls="--", lw=1)
    ax.set_yticks(range(len(SCENARIOS)), ["core-biased", "uniform (null)", "periphery-biased"])
    ax.set_xlabel("Excess E = $B_h$ − null mean (100 seeds)")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(RESULTS / f"ext_scenarios.{ext}", dpi=300)
    plt.close(fig)

    pd.set_option("display.width", 200)
    print(sc_df.drop(columns=["k", "r", "pct"]).round(4).to_string(index=False))
    print(cal_sum.round(4).to_string())
    print(sens.round(4).to_string(index=False))
    print(wdf.round(3).to_string(index=False))
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()
